**VALUASI EKONOMI PM2.5 PER PROVINSI (2010-2025)**

Menggabungkan:
  1. Data GBD (DALY & Deaths attributable PM2.5), 2010-2023
  2. Pengisian celah 2024-2025 (rate 2023 dipertahankan x populasi)
  3. Validasi CDR nasional (justifikasi asumsi rate stabil)
  4. Valuasi DALY-based (metode SENSITIVITAS -- konvensi WHO-CHOICE lama,
     tidak lagi direkomendasikan formal oleh WHO sejak Bertram dkk. 2016)
  5. Valuasi VSL-based (METODE UTAMA), formula & parameter dari
     World Bank (2016), "The Cost of Air Pollution: Strengthening the
     Economic Case for Action", halaman 48-49, Persamaan (3.1) --
     terverifikasi langsung terhadap teks asli, termasuk perbaikan
     konversi PDRB per kapita ke basis 2011 USD PPP resmi (bukan kurs pasar)
  6. Sanity-check terhadap benchmark nasional Indonesia dari laporan
     yang sama, halaman 95-96 (5,15% PDB, tahun 2013, ambient PM2.5)

Rujukan:
  - Apte, Marshall, Cohen, & Brauer (2015). "Addressing Global Mortality
    from Ambient PM2.5." Environmental Science & Technology.
  - Mathers & Loncar (2006). "Projections of Global Mortality and
    Burden of Disease from 2002 to 2030." PLoS Medicine.
  - World Bank (2016). "The Cost of Air Pollution: Strengthening the
    Economic Case for Action.

In [ ]:
import pandas as pd
import re
from google.colab import drive

drive.mount('/content/drive')
FOLDER_SUMBER = "/content/drive/MyDrive/DatabaseSkripsi/SumberData"
FOLDER_OUTPUT = "/content/drive/MyDrive/DatabaseSkripsi/Outputs"

Mounted at /content/drive


In [ ]:
# =====================================================================
# BAGIAN 1: DAFTAR PROVINSI STANDAR & HARMONISASI
# =====================================================================
PROVINSI_STANDAR = [
    "ACEH", "BALI", "BANTEN", "BENGKULU", "DI YOGYAKARTA", "DKI JAKARTA",
    "GORONTALO", "JAMBI", "JAWA BARAT", "JAWA TENGAH", "JAWA TIMUR",
    "KALIMANTAN BARAT", "KALIMANTAN SELATAN", "KALIMANTAN TENGAH",
    "KALIMANTAN TIMUR", "KALIMANTAN UTARA", "KEP. BANGKA BELITUNG",
    "KEP. RIAU", "LAMPUNG", "MALUKU", "MALUKU UTARA",
    "NUSA TENGGARA BARAT", "NUSA TENGGARA TIMUR", "PAPUA", "PAPUA BARAT",
    "RIAU", "SULAWESI BARAT", "SULAWESI SELATAN", "SULAWESI TENGAH",
    "SULAWESI TENGGARA", "SULAWESI UTARA", "SUMATERA BARAT",
    "SUMATERA SELATAN", "SUMATERA UTARA",
]

PEMETAAN_KHUSUS = {
    "DAERAH ISTIMEWA YOGYAKARTA": "DI YOGYAKARTA",
    "YOGYAKARTA": "DI YOGYAKARTA",
    "KEPULAUAN BANGKA BELITUNG": "KEP. BANGKA BELITUNG",
    "KEPULAUAN BANGKA-BELITUNG": "KEP. BANGKA BELITUNG",
    "KEPULAUAN RIAU": "KEP. RIAU",
    "JAKARTA RAYA": "DKI JAKARTA",
    "JAKARTA": "DKI JAKARTA",
    "SUMATRA BARAT": "SUMATERA BARAT",
    "SUMATRA SELATAN": "SUMATERA SELATAN",
    "SUMATRA UTARA": "SUMATERA UTARA",
}


def normalisasi_dasar(nama):
    return re.sub(r"\s+", " ", str(nama).strip())


def harmonisasi_provinsi(nama):
    nama_upper = normalisasi_dasar(nama).upper()
    if nama_upper in PEMETAAN_KHUSUS:
        return PEMETAAN_KHUSUS[nama_upper]
    for standar in PROVINSI_STANDAR:
        if nama_upper == standar.upper():
            return standar
    return None  # termasuk "INDONESIA" (level nasional)


# =====================================================================
# BAGIAN 2: BACA & HARMONISASI DATA GBD (DALY & DEATHS)
# =====================================================================
gbd = pd.read_csv(f"{FOLDER_SUMBER}/CDR_pollution_ihme.csv")
gbd = gbd[gbd["metric_name"].isin(["Angka", "Laju"])]
gbd = gbd[gbd["measure_name"].isin(["Meninggal", "DALYs (Disability-Adjusted Life Years)"])]
gbd["PROVINSI"] = gbd["location_name"].apply(harmonisasi_provinsi)
gbd = gbd[gbd["PROVINSI"].notnull()]  # buang "Indonesia" (nasional)

gbd_pivot = gbd.pivot_table(
    index=["PROVINSI", "year"], columns=["measure_name", "metric_name"], values="val",
).reset_index()
gbd_pivot.columns = [
    "_".join(c).strip("_") if isinstance(c, tuple) else c for c in gbd_pivot.columns
]
gbd_pivot = gbd_pivot.rename(columns={
    "Meninggal_Angka": "kematian_jumlah",
    "Meninggal_Laju": "kematian_rate_per100rb",
    "DALYs (Disability-Adjusted Life Years)_Angka": "daly_jumlah",
    "DALYs (Disability-Adjusted Life Years)_Laju": "daly_rate_per100rb",
})
print(f"[GBD] {gbd_pivot['PROVINSI'].nunique()} provinsi, "
      f"tahun {gbd_pivot['year'].min()}-{gbd_pivot['year'].max()}, "
      f"{len(gbd_pivot)} baris (harapan 34x14=476)")

# =====================================================================
# BAGIAN 3: BACA & HARMONISASI POPULASI + PDRB PER KAPITA
# =====================================================================
penduduk = pd.read_csv(
    f"{FOLDER_SUMBER}/Jumlah_Penduduk.csv",
    sep=";", decimal=",", encoding="utf-8-sig"
)
penduduk = penduduk.loc[:, ~penduduk.columns.str.contains("^Unnamed")]  # buang kolom kosong akibat ';' ekstra di akhir header
penduduk["PROVINSI"] = penduduk["Provinsi"].apply(harmonisasi_provinsi)
penduduk = penduduk.rename(columns={"Tahun": "year", "Penduduk": "populasi"})[
    ["PROVINSI", "year", "populasi"]
]
print(f"[Populasi] {penduduk['PROVINSI'].nunique()} provinsi (harapan 34)")

pdrb_kapita = pd.read_csv(
    f"{FOLDER_SUMBER}/PDRB_per_Kapita_ADHK.csv",
    sep=",", decimal=",", encoding="utf-8-sig"
)
pdrb_kapita.columns = ["PROVINSI", "year", "pdrb_per_kapita_ribu_rp"]

pdrb_kapita["pdrb_per_kapita_ribu_rp"] = pd.to_numeric(
    pdrb_kapita["pdrb_per_kapita_ribu_rp"], errors="coerce"
)

kosong_pdrb = pdrb_kapita[pdrb_kapita["pdrb_per_kapita_ribu_rp"].isnull()]
if len(kosong_pdrb) > 0:
    print(f"\n[PDRB per kapita] {len(kosong_pdrb)} baris kosong/tidak terparse -- KETERANGAN:")
    print(kosong_pdrb[["PROVINSI", "year"]].to_string(index=False))
    print(
        "CATATAN: kekosongan ini diduga terkait pemekaran Provinsi Kalimantan\n"
        "Utara (resmi terbentuk 25 Okt 2012, UU No. 20/2012) -- data PDRB per\n"
        "kapita Kalimantan Utara TIDAK ADA untuk tahun sebelum provinsi ini\n"
        "berdiri (2010-2011), dan Kalimantan Timur/Kalimantan Utara tahun 2012\n"
        "(tahun transisi) sering tertunda revisi resminya oleh BPS. Baris ini\n"
        "dibiarkan NaN (bukan diisi 0 atau interpolasi) dan otomatis akan\n"
        "dikecualikan dari estimasi GMM sbg bagian dari unbalanced panel --\n"
        "konsisten dgn pendekatan robustness with/without Kalimantan Utara\n"
        "yang sudah diterapkan pada model utama."
    )

BARIS_HARUS_NAN_PEMEKARAN = [
    ("KALIMANTAN UTARA", 2010), ("KALIMANTAN UTARA", 2011),
    ("KALIMANTAN UTARA", 2012), ("KALIMANTAN TIMUR", 2012),
]
for prov, thn in BARIS_HARUS_NAN_PEMEKARAN:
    mask_override = (pdrb_kapita["PROVINSI"] == prov) & (pdrb_kapita["year"] == thn)
    nilai_lama = pdrb_kapita.loc[mask_override, "pdrb_per_kapita_ribu_rp"]
    if mask_override.any() and nilai_lama.notnull().any():
        print(f"[PDRB per kapita] OVERRIDE ke NaN: {prov} {thn} "
              f"(nilai asal di sumber: {nilai_lama.values[0]:,.2f} ribu Rp -- "
              f"tidak wajar/tidak seharusnya ada data krn gap pemekaran provinsi)")
    pdrb_kapita.loc[mask_override, "pdrb_per_kapita_ribu_rp"] = None

# --- Deteksi outlier
median_per_tahun = pdrb_kapita.groupby("year")["pdrb_per_kapita_ribu_rp"].transform("median")
outlier = pdrb_kapita[
    pdrb_kapita["pdrb_per_kapita_ribu_rp"].notnull()
    & (pdrb_kapita["pdrb_per_kapita_ribu_rp"] > 20 * median_per_tahun)
]
if len(outlier) > 0:
    print(f"\n[PDRB per kapita] PERINGATAN -- {len(outlier)} baris terdeteksi OUTLIER "
          f"(>20x median provinsi lain di tahun yg sama), cek manual ke file sumber:")
    print(outlier[["PROVINSI", "year", "pdrb_per_kapita_ribu_rp"]].to_string(index=False))
else:
    print("[PDRB per kapita] Tidak ada outlier terdeteksi (>20x median tahun berjalan).")

# =====================================================================
# BAGIAN 4: GABUNGKAN + ISI CELAH 2024-2025 (rate 2023 x populasi)
# =====================================================================
panel = penduduk.merge(pdrb_kapita, on=["PROVINSI", "year"], how="left")
panel = panel.merge(gbd_pivot, on=["PROVINSI", "year"], how="left")

rate_2023 = gbd_pivot[gbd_pivot["year"] == 2023][
    ["PROVINSI", "daly_rate_per100rb", "kematian_rate_per100rb"]
].rename(columns={
    "daly_rate_per100rb": "daly_rate_2023", "kematian_rate_per100rb": "kematian_rate_2023",
})

for tahun_target in [2024, 2025]:
    mask = panel["year"] == tahun_target
    pt = panel[mask].merge(rate_2023, on="PROVINSI", how="left")
    panel.loc[mask, "daly_jumlah"] = pt["daly_rate_2023"].values * pt["populasi"].values / 100000
    panel.loc[mask, "kematian_jumlah"] = pt["kematian_rate_2023"].values * pt["populasi"].values / 100000

panel["sumber_gbd"] = panel["year"].apply(
    lambda y: "GBD_langsung" if y <= 2023 else "estimasi_rate_2023_x_populasi"
)

kosong = panel["daly_jumlah"].isnull().sum()
print(f"[Celah] {kosong} baris masih kosong setelah pengisian (harapan: 0)")

# =====================================================================
# BAGIAN 5: VALIDASI CDR
# =====================================================================
try:
    cdr = pd.read_csv(
        f"{FOLDER_SUMBER}/CDR_Indonesia.csv",
        sep=";", decimal=",", encoding="utf-8-sig"
    )
    cdr.columns = ["year", "cdr"]
    baseline_prapandemi = cdr[cdr["year"].between(2010, 2019)]["cdr"].mean()
    cdr_terkini = cdr[cdr["year"].between(2022, 2025)]["cdr"].mean()
    selisih_persen = abs(cdr_terkini - baseline_prapandemi) / baseline_prapandemi * 100

    print(f"\n[Validasi CDR] Rata-rata CDR 2010-2019 (pra-pandemi): {baseline_prapandemi:.3f}")
    print(f"[Validasi CDR] Rata-rata CDR 2022-2025 (pasca-pandemi): {cdr_terkini:.3f}")
    print(f"[Validasi CDR] Selisih: {selisih_persen:.2f}%")
    if selisih_persen < 5:
        print("[Validasi CDR] KESIMPULAN: kondisi mortalitas nasional 2022-2025 "
              "sudah kembali stabil mendekati level pra-pandemi -- mendukung "
              "asumsi rate attributable PM2.5 yang relatif stabil pada periode "
              "ekstrapolasi 2024-2025 (lihat Apte et al. 2015; Mathers & Loncar 2006).")
    else:
        print("[Validasi CDR] PERHATIAN: ada deviasi CDR >5% dari baseline -- "
              "pertimbangkan kembali asumsi rate konstan.")
except FileNotFoundError:
    print("\n[Validasi CDR] File CDR_Indonesia.xlsx tidak ditemukan -- validasi dilewati.")

# =====================================================================
# BAGIAN 6: VALUASI DALY x PDB PER KAPITA (SENSITIVITAS)
#    Konvensi 1x/3x PDB per kapita per DALY (WHO Commission on
#    Macroeconomics and Health, 2001 / WHO-CHOICE)
# =====================================================================
FAKTOR_SATUAN_PDRB = 1000  # asumsi: ribu Rupiah -> Rupiah penuh (VERIFIKASI!)

panel["nilai_per_daly_1x"] = panel["pdrb_per_kapita_ribu_rp"] * FAKTOR_SATUAN_PDRB * 1
panel["nilai_per_daly_3x"] = panel["pdrb_per_kapita_ribu_rp"] * FAKTOR_SATUAN_PDRB * 3
panel["nilai_pm25_rp_daly_1x"] = panel["daly_jumlah"] * panel["nilai_per_daly_1x"]
panel["nilai_pm25_rp_daly_3x"] = panel["daly_jumlah"] * panel["nilai_per_daly_3x"]

# =====================================================================
# BAGIAN 7: VALUASI VSL-BASED (METODE UTAMA)
#    Formula & parameter: World Bank (2016), "The Cost of Air Pollution:
#    Strengthening the Economic Case for Action", hal. 48-49, Persamaan (3.1):
#    VSL_c,n = VSL_OECD x (Y_c,n / Y_OECD) ^ e
#
#    - VSL_OECD = USD 3,83 juta -- "base VSL of $3.83 million"
#    - Y_OECD   = USD 37.000    -- "average GDP per capita of about $37,000"
#    - Elastisitas: 1,2 untuk negara berpendapatan
#      rendah-menengah (range 1,0-1,4) vs 0,8 untuk negara berpendapatan
#      tinggi (range 0,6-1,0). Indonesia = lower-middle-income
#    - Rasio Y_c/Y_OECD memakai basis PPP 2011
#
#    Dipilih sebagai metode utama karena: (1) basis WTP/welfare economics
#    lebih kuat drpd konvensi DALYxPDB yg sudah tidak direkomendasikan formal
#    oleh WHO sejak Bertram dkk. (2016); (2) arah yg diambil literatur
#    Indonesia terkini (Syuhada dkk., 2023)
# =====================================================================
VSL_OECD_USD = 3_830_000       # USD 3,83 juta, basis 2011 USD PPP (World Bank 2016, hal. 48)
Y_OECD_USD = 37_000            # USD 37.000, basis 2011 USD PPP (World Bank 2016, hal. 48)
ELASTISITAS = 1.2              # LMIC (World Bank 2016, hal. 49) -- Indonesia = lower-middle-income

#     Sumber: World Bank WDI, GDP deflator Indonesia (NY.GDP.DEFL.ZS):
#     2010 = 100,00 ; 2011 = 107,47
DEFLATOR_ID_2010 = 100.00
DEFLATOR_ID_2011 = 107.47
FAKTOR_GESER_BASIS_2011 = DEFLATOR_ID_2011 / DEFLATOR_ID_2010

#     PA.NUS.PPP) = Rp 3.512,75 / international-$ -- BUKAN kurs pasar
#     (~Rp 8.779/USD th 2011), sesuai syarat eksplisit formula (3.1) di atas.
PPP_CONVERSION_2011 = 3512.75  # Rupiah per international dollar, tahun 2011 (World Bank WDI)

panel["pdrb_per_kapita_rp"] = panel["pdrb_per_kapita_ribu_rp"] * FAKTOR_SATUAN_PDRB
panel["pdrb_per_kapita_rp_basis2011"] = panel["pdrb_per_kapita_rp"] * FAKTOR_GESER_BASIS_2011
panel["pdrb_per_kapita_usd_ppp2011"] = panel["pdrb_per_kapita_rp_basis2011"] / PPP_CONVERSION_2011

panel["vsl_provinsi_usd_ppp2011"] = VSL_OECD_USD * (
    panel["pdrb_per_kapita_usd_ppp2011"] / Y_OECD_USD
) ** ELASTISITAS

panel["vsl_provinsi_rp_basis2011"] = panel["vsl_provinsi_usd_ppp2011"] * PPP_CONVERSION_2011
panel["vsl_provinsi_rp"] = panel["vsl_provinsi_rp_basis2011"] / FAKTOR_GESER_BASIS_2011

panel["nilai_pm25_rp_vsl"] = panel["kematian_jumlah"] * panel["vsl_provinsi_rp"]

# CATATAN KETERBATASAN: PPP conversion factor & deflator di atas adalah
# angka NASIONAL Indonesia (tidak ada PPP conversion factor resmi per
# provinsi), diterapkan seragam ke semua provinsi

# =====================================================================
# BAGIAN 8: SANITY-CHECK TERHADAP BENCHMARK WORLD BANK (~5% PDB, 2013)
#    Memakai PDRB ADHK RESMI
# =====================================================================
pdrb_adhk = pd.read_csv(
    f"{FOLDER_SUMBER}/PDRB_ADHK_Rill.csv",
    sep=";", decimal=",", encoding="utf-8-sig"
)
pdrb_adhk.columns = ["PROVINSI", "year", "pdrb_adhk_rp"]
pdrb_adhk["PROVINSI"] = pdrb_adhk["PROVINSI"].apply(harmonisasi_provinsi)

panel = panel.merge(pdrb_adhk, on=["PROVINSI", "year"], how="left")

cek_2013 = panel[panel["year"] == 2013]
if len(cek_2013) > 0 and cek_2013["pdrb_adhk_rp"].notnull().any():
    rasio_daly = cek_2013["nilai_pm25_rp_daly_1x"].sum() / cek_2013["pdrb_adhk_rp"].sum() * 100
    rasio_vsl = cek_2013["nilai_pm25_rp_vsl"].sum() / cek_2013["pdrb_adhk_rp"].sum() * 100
    print(f"\n[Sanity-check 2013, PDRB ADHK RESMI] Nilai PM2.5 (VSL, UTAMA)      / PDRB ADHK: {rasio_vsl:.2f}%")
    print(f"[Sanity-check 2013, PDRB ADHK RESMI] Nilai PM2.5 (DALY-1x, sensitivitas) / PDRB ADHK: {rasio_daly:.2f}%")
    print(f"[Sanity-check 2013] Benchmark World Bank (2016, hal. 95-96): 5,15% PDB")
    print("[Sanity-check] Kalau rasio Anda jauh berbeda (misal >20% atau <0,5%), "
          "cek ulang satuan PDRB per kapita atau FAKTOR_SATUAN_PDRB.")
else:
    print("\n[Sanity-check] Tidak bisa dihitung -- cek kecocokan PROVINSI/year dgn PDRB_ADHK_Rill.csv.")

# =====================================================================
# BAGIAN 9: VERIFIKASI & SIMPAN
# =====================================================================
print(f"\n{'='*70}\nRINGKASAN AKHIR\n{'='*70}")
print(f"Jumlah baris   : {len(panel)} (harapan 34x16=544)")
print(f"Jumlah provinsi: {panel['PROVINSI'].nunique()}")

print("\nContoh hasil (Aceh):")
kolom_tampil = ["year", "daly_jumlah", "kematian_jumlah", "nilai_pm25_rp_daly_1x",
                 "nilai_pm25_rp_vsl", "sumber_gbd"]
print(panel[panel["PROVINSI"] == "ACEH"][kolom_tampil].to_string(
    index=False, float_format=lambda x: f"{x:,.0f}"))

output_path = f"{FOLDER_OUTPUT}/panel_nilai_pm25_lengkap_2010_2025.csv"
panel.to_csv(output_path, index=False)
print(f"\nDisimpan ke: {output_path}")
print(
    "\nCATATAN METODOLOGI:\n"
    "- nilai_pm25_rp_vsl     = metode UTAMA (VSL, formula & parameter World\n"
    "  Bank 2016, hal. 48-49, Persamaan 3.1 -- VSL_OECD=$3,83jt, Y_OECD=$37rb,\n"
    "  elastisitas=1,2 utk LMIC -- semua terverifikasi persis sesuai teks asli).\n"
    "  PDRB per kapita dikonversi ke basis 2011 USD PPP (sesuai syarat\n"
    "  eksplisit formula), memakai GDP deflator Indonesia (100,00->107,47)\n"
    "  dan PPP conversion factor resmi World Bank WDI (Rp 3.512,75/int'l-$,\n"
    "  2011) -- BUKAN kurs pasar. Keterbatasan: PPP conversion factor yg\n"
    "  dipakai adalah angka nasional (tidak ada data PPP per provinsi),\n"
    "  diterapkan seragam ke semua provinsi.\n"
    "- nilai_pm25_rp_daly_1x = sensitivitas (1x PDRB per kapita per DALY,\n"
    "  konvensi WHO-CHOICE lama -- tidak lagi direkomendasikan formal oleh\n"
    "  WHO sejak Bertram dkk. 2016, tapi masih lazim dipakai di literatur).\n"
    "- nilai_pm25_rp_daly_3x = sensitivitas atas (3x PDRB per kapita).\n"
    "- sumber_gbd membedakan data GBD asli (2010-2023) dari estimasi\n"
    "  perluasan (2024-2025, rate 2023 x populasi berjalan).\n"
    "- Validasi CDR mendukung asumsi rate stabil untuk perluasan tsb."
)


[GBD] 34 provinsi, tahun 2010-2023, 476 baris (harapan 34x14=476)
[Populasi] 34 provinsi (harapan 34)

[PDRB per kapita] 3 baris kosong/tidak terparse -- KETERANGAN:
        PROVINSI  year
KALIMANTAN UTARA  2010
KALIMANTAN UTARA  2011
KALIMANTAN UTARA  2012
CATATAN: kekosongan ini diduga terkait pemekaran Provinsi Kalimantan
Utara (resmi terbentuk 25 Okt 2012, UU No. 20/2012) -- data PDRB per
kapita Kalimantan Utara TIDAK ADA untuk tahun sebelum provinsi ini
berdiri (2010-2011), dan Kalimantan Timur/Kalimantan Utara tahun 2012
(tahun transisi) sering tertunda revisi resminya oleh BPS. Baris ini
dibiarkan NaN (bukan diisi 0 atau interpolasi) dan otomatis akan
dikecualikan dari estimasi GMM sbg bagian dari unbalanced panel --
konsisten dgn pendekatan robustness with/without Kalimantan Utara
yang sudah diterapkan pada model utama.
[PDRB per kapita] OVERRIDE ke NaN: KALIMANTAN TIMUR 2012 (nilai asal di sumber: 124,501.88 ribu Rp -- tidak wajar/tidak seharusnya ada data krn gap pemekaran pr